In [1]:
import pandas as pd
import utils
from Bio import SeqIO
import pickle
import argparse
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from transformers import T5EncoderModel, T5Tokenizer
from config import config
from dataset import MyDataset
from torch import optim, nn
from torch.utils.data import TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.ensemble import RandomForestClassifier
from model import Cnn
from loss import *
from torch.utils.data import DataLoader
from torch.optim import lr_scheduler
from utils import *
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, matthews_corrcoef
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
utils.seed_everything(config.seed)

# Binary (PVP Indentification)

In [3]:
label_dict = {'non-PVP': 0, 'PVP': 1}
test_df = pd.read_csv(config.binary_test_data)
test_proteins, test_ids, test_labels = [], [], []

for index, row in test_df.iterrows():
    test_proteins.append(row['sequence'])
    test_ids.append(row['accession'])
    test_labels.append(label_dict[row['label']])
    
y_test = np.array(test_labels)
test_data = MyDataset(test_ids, test_labels)
test_dataloader = DataLoader(test_data, shuffle=False, batch_size=config.batch_size)

In [4]:
# get embedding
embedding_df = pd.read_excel(config.embedding_file)
embedding_df.set_index("id", inplace=True)

In [5]:
model = Cnn().to(config.device)
model.load_state_dict(torch.load(config.binary_model))
# test
labels, test_epoch_preds = test(model, test_dataloader, embedding_df)
# results
test_acc = accuracy_score(y_test, test_epoch_preds)
precision = precision_score(y_test, test_epoch_preds)
recall = recall_score(y_test, test_epoch_preds)
f1 = f1_score(y_test, test_epoch_preds)
mcc = matthews_corrcoef(y_test, test_epoch_preds)
tn, fp, fn, tp = confusion_matrix(y_test, test_epoch_preds).ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
print('test ACC of classifier: %.4f' % (test_acc))
print('test precision of classifier: %.4f' % (precision))
print('test recall of classifier: %.4f' % (recall))
print('test f1 of classifier: %.4f' % (f1))
print('test mcc of classifier: %.4f' % (mcc))
print('test sensitivity of classifier: %.4f' % (sensitivity))
print('test specificity of classifier: %.4f' % (specificity))

/home/oyh/anaconda3/envs/pytorch/lib/python3.8/site-packages/torch/nn/modules/lazy.py:178: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


test ACC of classifier: 0.9717
test precision of classifier: 0.9528
test recall of classifier: 0.9823
test f1 of classifier: 0.9673
test mcc of classifier: 0.9427
test sensitivity of classifier: 0.9823
test specificity of classifier: 0.9638


# Multi-class (PVP function annotation)

In [6]:
label_dict = {'minor capsid':0, 'tail fiber':1, 'major tail':2, 'portal':3, 'minor tail':4, 'baseplate':5, 'major capsid':6}
test_df = pd.read_csv(config.test_muti_data)
test_proteins, test_ids, test_labels = [], [], []
for index, row in test_df.iterrows():
    test_proteins.append(row['sequence'])
    test_ids.append(row['accession'])
    test_labels.append(label_dict[row['label']])

y_test = np.array(test_labels)
test_data = MyDataset(test_ids, test_labels)
test_dataloader = DataLoader(test_data, shuffle=False, batch_size=config.batch_size)

In [7]:
model = Cnn_muti().to(config.device)
model.load_state_dict(torch.load(config.muti_model))
# test
labels, test_epoch_preds = test_muti(model, test_dataloader, embedding_df)

# results
test_acc = accuracy_score(y_test, test_epoch_preds)
precision = precision_score(y_test, test_epoch_preds, average='weighted')
recall = recall_score(y_test, test_epoch_preds, average='weighted')
f1 = f1_score(y_test, test_epoch_preds, average='weighted')
mcc = matthews_corrcoef(y_test, test_epoch_preds)
print('test ACC of classifier: %.4f' % (test_acc))
print('test precision of classifier: %.4f' % (precision))
print('test recall of classifier: %.4f' % (recall))
print('test f1 of classifier: %.4f' % (f1))
print('test mcc of classifier: %.4f' % (mcc))

test_precision = precision_score(y_test, test_epoch_preds, average=None)
test_recall = recall_score(y_test, test_epoch_preds, average=None)
test_f1 = f1_score(y_test, test_epoch_preds, average=None)
name = list(label_dict.keys())
df = pd.DataFrame({'family host': name, 'Precision': test_precision, 'Recall': test_recall, 'F1': test_f1})
df.to_csv(f'results/split_time/muti_class_predict_results.csv', index=False)

/home/oyh/anaconda3/envs/pytorch/lib/python3.8/site-packages/torch/nn/modules/lazy.py:178: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


test ACC of classifier: 0.9679
test precision of classifier: 0.9707
test recall of classifier: 0.9679
test f1 of classifier: 0.9686
test mcc of classifier: 0.9600
